# Whisper Evaluation (E1)


CallSense — Speech Pipeline & Whisper Comparison (E1)

يهدف هذا النوتبوك إلى تجهيز البيانات الصوتية ومقارنة أداء ثلاثة نماذج من Whisper (Tiny, Base, Small) في تحويل الكلام العربي إلى نص، باستخدام مقياس Word Error Rate (WER) لتقييم دقة النتائج.

مصادر البيانات:

Common Voice Arabic: 50 تسجيلًا.
MGB-2 Arabic: 100 تسجيل.
تسجيلات الفريق: 50 تسجيلًا.

إجمالي البيانات المتوقع: 200 تسجيل صوتي.

In [1]:
# CELL 01
# نحدد بيئة Python ومسار سطح المكتب
from pathlib import Path
import sys

# هذا المسار المعتاد لسطح المكتب، وتقدرين تغيّرينه هنا بس
DESKTOP = Path.home() / "Desktop"

print("Python:", sys.version.split()[0])
print("بيئة Jupyter:", sys.executable)
print("مسار Desktop:", DESKTOP)
print("هل Desktop موجود؟", DESKTOP.is_dir())

Python: 3.11.16
بيئة Jupyter: c:\Users\user\miniconda3\envs\callsense\python.exe
مسار Desktop: C:\Users\user\Desktop
هل Desktop موجود؟ True


#   فحص البيئة والبيانات

في هذا القسم نتحقق من بيئة Python والمكتبات المطلوبة، ونتأكد من توفر FFmpeg. بعدها نحدد مصادر البيانات الثلاثة ونتحقق من وجود الملفات الصوتية والنصوص المرتبطة بها، عشان تكون البيانات جاهزة لتجارب Whisper.

In [2]:
# CELL 02 | نتأكد من وجود مجلد سطح المكتب

if not DESKTOP.is_dir():
    raise FileNotFoundError("مجلد Desktop غير موجود، راجعي المسار في CELL 01.")

print("تم التأكد من وجود مجلد Desktop بنجاح.")

تم التأكد من وجود مجلد Desktop بنجاح.


تجهيز المكتبات

في هالخطوة نتأكد من توفر المكتبات اللازمة لتشغيل Whisper وتقييم نتائجه. إذا فيه مكتبة ناقصة، نحاول تثبيتها داخل بيئة Python المستخدمة في النوتبوك.

In [3]:
# CELL 03 | نفحص المكتبات ونثبّت الناقص منها فقط
import importlib.util
import subprocess

packages = {
    "pandas": "pandas",
    "torch": "torch",
    "whisper": "openai-whisper",
    "jiwer": "jiwer",
}
missing = [pkg for module, pkg in packages.items()
           if importlib.util.find_spec(module) is None]

if missing:
    print("بنثبّت المكتبات الناقصة فقط:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("✅ كل المكتبات موجودة")

✅ كل المكتبات موجودة


In [4]:
# CELL 04 | نستورد المكتبات ونتأكد إنها تشتغل

import pandas as pd
import torch
import whisper
import jiwer
from IPython.display import display
from importlib.metadata import version

print("pandas:", pd.__version__)
print("PyTorch:", torch.__version__)
print("Whisper:", version("openai-whisper"))
print("jiwer:", version("jiwer"))

pandas: 3.0.6
PyTorch: 2.14.1+cpu
Whisper: 20250625
jiwer: 4.0.0


In [5]:
# CELL 05 | نتأكد إن FFmpeg يشتغل من داخل Jupyter

import shutil
import subprocess

FFMPEG = shutil.which("ffmpeg")

if FFMPEG is None:
    print("❌ FFmpeg مو ظاهر داخل Jupyter.")
    print("أعيدي تشغيل VS Code بالكامل، ثم شغّلي الخلية مرة ثانية.")
else:
    check = subprocess.run(
        [FFMPEG, "-version"],
        capture_output=True,
        text=True
    )

    if check.returncode == 0:
        print("✅ FFmpeg شغال بنجاح")
        print(check.stdout.splitlines()[0])
    else:
        print("❌ لقينا FFmpeg، لكن تشغيله فشل.")

✅ FFmpeg شغال بنجاح
ffmpeg version 9.0.2-full_build-www.gyan.dev Copyright (c) 2000-2026 the FFmpeg developers


In [6]:
# CELL 06 | إعدادات مصادر البيانات الثلاثة

SOURCES = {
    "CommonVoice": {
        "folder": r"C:\Users\user\Desktop\CommonVoice_Arabic_v27\commonvoice_sample_50",
        "csv": "",
        "audio_col": "",
        "text_col": "",
        "expected": 50,
    },
    "MGB2": {
        "folder": r"C:\Users\user\Desktop\MGB2\mgb2_sample_100",
        "csv": "",
        "audio_col": "",
        "text_col": "",
        "expected": 100,
    },
    "Team": {
        "folder": r"C:\Users\user\Desktop\team_recording\team_recording",
        "csv": "CallSense_plan - Calls.csv",
        "audio_col": "audio_file",
        "text_col": "transcript",
        "expected": 50,
    },
}

folder_hints = {
    "CommonVoice": ("common", "voice"),
    "MGB2": ("mgb",),
    "Team": ("team", "فريق", "recordings", "تسجيلات"),
}

In [7]:
# CELL 07 | نتحقق من وجود مجلدات مصادر البيانات

source_folders = {}

for name, settings in SOURCES.items():
    folder = Path(settings["folder"]).expanduser()

    if not folder.is_dir():
        raise FileNotFoundError(
            f"❌ مجلد {name} غير موجود:\n{folder}"
        )

    source_folders[name] = folder.resolve()
    print(f"✅ {name}: {source_folders[name]}")

print("\nتم التحقق من مجلدات مصادر البيانات الثلاثة.")

✅ CommonVoice: C:\Users\user\Desktop\CommonVoice_Arabic_v27\commonvoice_sample_50
✅ MGB2: C:\Users\user\Desktop\MGB2\mgb2_sample_100
✅ Team: C:\Users\user\Desktop\team_recording\team_recording

تم التحقق من مجلدات مصادر البيانات الثلاثة.


In [8]:
# CELL 08 | نكتشف ملفات CSV والصوت لكل مصدر
AUDIO_EXT = {".wav", ".mp3", ".m4a", ".flac", ".ogg", ".opus", ".aac", ".webm", ".wma", ".mp4"}
source_files = {}

for name, folder in source_folders.items():
    files = [p for p in folder.rglob("*") if p.is_file()]
    csvs = [p for p in files if p.suffix.lower() == ".csv"]
    audios = [p for p in files if p.suffix.lower() in AUDIO_EXT]
    source_files[name] = {"csvs": csvs, "audios": audios}
    print(f"{name}: لقينا {len(csvs)} CSV و{len(audios)} ملف صوت")
    print("ملفات CSV:", [str(p.relative_to(folder)) for p in csvs])
    if not audios:
        raise ValueError(f"ما لقينا ملفات صوت في {name}. شيّكي folder في CELL 06.")

CommonVoice: لقينا 1 CSV و50 ملف صوت
ملفات CSV: ['speech_metadata.csv']
MGB2: لقينا 1 CSV و100 ملف صوت
ملفات CSV: ['speech_metadata.csv']
Team: لقينا 2 CSV و49 ملف صوت
ملفات CSV: ['callSense_plan - Calls.csv', 'callSense_plan - Plan.csv']


In [9]:
# CELL 09 | قراءة ملفات البيانات والتحقق منها

tables = {}
csv_paths = {}

for name, settings in SOURCES.items():
    folder = source_folders[name]

    # تحديد ملف CSV
    if settings["csv"]:
        csv_file = Path(settings["csv"])
        if not csv_file.is_absolute():
            csv_file = folder / csv_file
    else:
        csv_files = source_files[name]["csvs"]

        if len(csv_files) != 1:
            raise ValueError(
                f"يجب تحديد ملف CSV الصحيح للمصدر: {name}"
            )

        csv_file = csv_files[0]

    # التحقق من وجود الملف
    if not csv_file.is_file():
        raise FileNotFoundError(
            f"لم يتم العثور على الملف: {csv_file}"
        )

    # قراءة البيانات
    tables[name] = pd.read_csv(
        csv_file,
        dtype=str,
        keep_default_na=False,
        encoding="utf-8-sig"
    )

    csv_paths[name] = csv_file.resolve()

    print(f"المصدر: {name}")
    print(f"عدد الصفوف: {len(tables[name])}")
    print(f"أسماء الأعمدة: {list(tables[name].columns)}")
    print("-" * 40)

print("اكتملت قراءة ملفات البيانات بنجاح.")

المصدر: CommonVoice
عدد الصفوف: 50
أسماء الأعمدة: ['audio_id', 'source', 'audio_path', 'transcript']
----------------------------------------
المصدر: MGB2
عدد الصفوف: 100
أسماء الأعمدة: ['audio_id', 'source', 'audio_path', 'transcript']
----------------------------------------
المصدر: Team
عدد الصفوف: 51
أسماء الأعمدة: ['call_id', 'audio_file', 'speaker_pair', 'dialect', 'intent', 'escalation', 'duration_sec', 'status', 'transcript', 'notes']
----------------------------------------
اكتملت قراءة ملفات البيانات بنجاح.


# الربط بين النص وملف الصوت

نتحقق من أسماء الأعمدة الموجودة في ملفات CSV. إذا لم نتعرف على عمود الصوت أو النص، نحدد اسمه في CELL 06 باستخدام الأسماء الظاهرة في CELL 09. 
بعدها نطابق مسارات الصوت مع الملفات الفعلية. إذا وجدنا ملفات متشابهة لا يمكن التمييز بينها، نتوقف بدلًا من اختيار ملف غير صحيح.

In [10]:
# CELL 10 | تحديد أعمدة الصوت والنص

chosen_columns = {
    "CommonVoice": {
        "audio": "audio_path",
        "text": "transcript"
    },
    "MGB2": {
        "audio": "audio_path",
        "text": "transcript"
    },
    "Team": {
        "audio": "audio_file",
        "text": "transcript"
    }
}

for name, columns in chosen_columns.items():
    for column in columns.values():
        if column not in tables[name].columns:
            raise ValueError(
                f"العمود {column} غير موجود في المصدر {name}"
            )

    print(
        f"{name} | عمود الصوت: {columns['audio']} "
        f"| عمود النص: {columns['text']}"
    )

print("تم التحقق من أعمدة المصادر الثلاثة بنجاح.")

CommonVoice | عمود الصوت: audio_path | عمود النص: transcript
MGB2 | عمود الصوت: audio_path | عمود النص: transcript
Team | عمود الصوت: audio_file | عمود النص: transcript
تم التحقق من أعمدة المصادر الثلاثة بنجاح.


# تجهيز النص العربي قبل WER

توحّد هذه الخطوة شكل الحروف، وتزيل التشكيل والتطويل وعلامات الترقيم، وتوحّد بعض الحروف العربية.

تحافظ المعالجة على اللهجات والكلمات الإنجليزية والأرقام، وتُطبّق بالطريقة نفسها على النص الأصلي وتفريغ Whisper.

In [11]:
# CELL 11 | تجهيز النص العربي

import re
import unicodedata

def normalize_ar(text):
    text = unicodedata.normalize("NFKC", str(text))
    text = re.sub(r"[ً-ٰٟۖ-ۭـ]", "", text)
    text = re.sub(r"[أإآٱ]", "ا", text).replace("ى", "ي")
    text = re.sub(r"[^\w\s]|_", " ", text)
    return " ".join(text.split())

print(normalize_ar("إلَى السُّوق، إن شاءَ الله!"))

الي السوق ان شاء الله


In [12]:
# CELL 12 | البحث عن ملف الصوت

def find_audio(value, folder, audio_files):
    raw = str(value).strip().replace("\\", "/")

    if not raw:
        return None

    path = Path(raw).expanduser()

    for candidate in (path, folder / path):
        if candidate.is_file() and candidate.suffix.lower() in AUDIO_EXT:
            return candidate.resolve()

    name = path.name.lower()
    stem = path.stem.lower().replace("-", "_")

    matches = [
        file for file in audio_files
        if file.name.lower() == name
        or file.stem.lower().replace("-", "_") == stem
        or file.stem.lower().replace("-", "_") == stem.replace("call_", "cal_", 1)
    ]

    if len(matches) == 1:
        return matches[0].resolve()

    return None

In [13]:
# CELL 13 | ربط التسجيلات بالنصوص وتسجيل المشكلات

records = []
problems = []

for source, table in tables.items():
    columns = chosen_columns[source]
    folder = source_folders[source]
    audio_files = source_files[source]["audios"]

    for i, row in table.iterrows():
        audio_name = str(row[columns["audio"]]).strip()
        reference = str(row[columns["text"]]).strip()

        # تجاهل الصف الفارغ بالكامل
        if audio_name.lower() in ("", "nan", "none") and reference.lower() in ("", "nan", "none"):
            continue

        audio_path = find_audio(
            audio_name,
            folder,
            audio_files
        )

        if audio_path is None or not normalize_ar(reference):
            problems.append({
                "المصدر": source,
                "صف CSV": i + 2,
                "اسم الصوت": audio_name,
                "النص موجود؟": bool(normalize_ar(reference)),
                "الصوت موجود؟": audio_path is not None
            })
            continue

        records.append({
            "source": source,
            "sample_id": f"{source}-{i+2:04d}",
            "audio_path": str(audio_path),
            "reference": reference
        })

data = pd.DataFrame(records)

print("التسجيلات المرتبطة بنجاح:", len(data))
print("السجلات التي تحتاج مراجعة:", len(problems))

if problems:
    display(pd.DataFrame(problems))

التسجيلات المرتبطة بنجاح: 199
السجلات التي تحتاج مراجعة: 2


,المصدر,صف CSV,اسم الصوت,النص موجود؟,الصوت موجود؟
0,Team,47,call_046.m4a,True,False
1,Team,52,call_051,False,False


In [14]:
# CELL 14 | التحقق من أعداد التسجيلات وعدم تكرار الصوت

expected_counts = {
    "CommonVoice": 50,
    "MGB2": 100,
    "Team": 49
}

for source, expected in expected_counts.items():
    actual = int((data["source"] == source).sum())
    print(f"{source}: {actual}/{expected}")

    if actual != expected:
        raise ValueError(f"عدد التسجيلات غير المتوقع في {source}")

if data["audio_path"].duplicated().any():
    display(data[data["audio_path"].duplicated(keep=False)])
    raise ValueError("يوجد ملف صوت مرتبط بأكثر من تسجيل.")

print("تم التحقق من الأعداد وعدم تكرار ملفات الصوت.")
print("ملاحظة: بيانات الفريق المتاحة 49 تسجيلًا، والتسجيل المستهدف المتبقي قيد التحقق.")

CommonVoice: 50/50
MGB2: 100/100
Team: 49/49
تم التحقق من الأعداد وعدم تكرار ملفات الصوت.
ملاحظة: بيانات الفريق المتاحة 49 تسجيلًا، والتسجيل المستهدف المتبقي قيد التحقق.


# فحص سلامة الملفات الصوتية

يُستخدم FFmpeg للتأكد من أن جميع التسجيلات قابلة للقراءة والمعالجة، وليس فقط أن ملفاتها موجودة. لا تُعدّل هذه الخطوة الملفات الأصلية، وتُعرض الملفات التي يتعذر فحصها للمراجعة.

In [15]:
# CELL 15 | نفحص فك ترميز كل تسجيل عن طريق FFmpeg
bad_audio = []

for i, audio_path in enumerate(data["audio_path"], start=1):
    try:
        result = subprocess.run(
            [FFMPEG, "-hide_banner", "-loglevel", "error", "-nostdin", "-i", audio_path,
             "-map", "0:a:0", "-f", "null", "-"],
            stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True, timeout=120)
        if result.returncode != 0:
            bad_audio.append((audio_path, result.stderr.strip()[:180]))
    except subprocess.TimeoutExpired:
        bad_audio.append((audio_path, "التحقق تجاوز 120 ثانية"))
    if i % 20 == 0 or i == len(data):
        print(f"فحصنا {i}/{len(data)} تسجيل")

if bad_audio:
    display(pd.DataFrame(bad_audio, columns=["ملف الصوت", "سبب المشكلة"]).head(10))
    raise ValueError("فيه ملفات صوت ما قدر FFmpeg يقرأها. صلحيها قبل ما تبدين Whisper.")
print("✅ كل التسجيلات انفتحت عن طريق FFmpeg")

فحصنا 20/199 تسجيل
فحصنا 40/199 تسجيل
فحصنا 60/199 تسجيل
فحصنا 80/199 تسجيل
فحصنا 100/199 تسجيل
فحصنا 120/199 تسجيل
فحصنا 140/199 تسجيل
فحصنا 160/199 تسجيل
فحصنا 180/199 تسجيل
فحصنا 199/199 تسجيل
✅ كل التسجيلات انفتحت عن طريق FFmpeg


# تجهيز مجلد النتائج

ننشىء مجلدًا لحفظ نتائج تجارب Whisper، ونحفظ فيه قائمة التسجيلات الجاهزة للتقييم. تبقى الملفات الصوتية الأصلية في أماكنها دون تعديل.


In [16]:
# CELL 16 | نختار مجلد النتائج ونحفظ قائمة التسجيلات الجاهزة

RESULTS_DIR = Path.cwd().parent / "results" / "whisper"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

data.to_csv(RESULTS_DIR / "ready_samples.csv", index=False, encoding="utf-8-sig")

print("✅ بنحفظ النتائج هنا:", RESULTS_DIR.resolve())

✅ بنحفظ النتائج هنا: C:\Users\user\Desktop\CallSense\results\whisper


In [17]:
# CELL 17 | اختيار جهاز المعالجة

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("جهاز المعالجة:", DEVICE.upper())

جهاز المعالجة: CPU


In [18]:
# CELL 18 | إنشاء بصمة لكل تسجيل
import hashlib

def make_key(row):
    path = Path(row.audio_path)

    info = path.stat()
    values = [
        row.source,
        row.sample_id,
        row.reference,
        str(info.st_size),
        str(info.st_mtime_ns),
        DEVICE
    ]

    return hashlib.sha256("||".join(values).encode()).hexdigest()

data["row_key"] = [make_key(row) for row in data.itertuples()]

print("تم تجهيز بصمة التسجيلات:", len(data))
#فائدة هذا الكود انه يعطي لكل تسجيل بصمه بحيث لو توقف البرنامج فجأه نقدر نعرف  التسجيلات اللي خلصنا منها 

تم تجهيز بصمة التسجيلات: 199


# Whisper Tiny

في هذا القسم، نستخدم نموذج Whisper Tiny لتفريغ 199 تسجيلًا صوتيًا باللغة العربية إلى نصوص مكتوبة. بعد ذلك، نحسب معدل الخطأ (WER) بمقارنة النصوص الناتجة بالنصوص الصحيحة، ونحفظ النتائج تدريجيًا حتى نتمكن من استكمال المعالجة إذا توقفت. ستُستخدم هذه النتائج لاحقًا لمقارنة أداء نماذج Whisper Tiny وBase وSmall.

In [19]:
# CELL 19 | تشغيل النموذج وحفظ النتائج

def run_model(model_name):
    output = RESULTS_DIR / f"whisper_{model_name}.csv"
    columns = ["row_key", "source", "audio_path",
               "reference", "hypothesis", "wer"]

    if output.exists():
        saved = pd.read_csv(output, keep_default_na=False)
    else:
        saved = pd.DataFrame(columns=columns)

    done = saved["row_key"].tolist()
    pending = data[~data["row_key"].isin(done)]

    print("التسجيلات المتبقية:", len(pending))

    if len(pending) > 0:
        model = whisper.load_model(model_name, device=DEVICE)

        for i, row in enumerate(pending.itertuples(), start=1):
            result = model.transcribe(
                row.audio_path,
                language="ar",
                fp16=(DEVICE == "cuda")
            )

            text_result = result["text"].strip()
            error = jiwer.wer(
                normalize_ar(row.reference),
                normalize_ar(text_result)
            )

            new_row = {
                "row_key": row.row_key,
                "source": row.source,
                "audio_path": row.audio_path,
                "reference": row.reference,
                "hypothesis": text_result,
                "wer": error
            }

            saved = pd.concat(
                [saved, pd.DataFrame([new_row])],
                ignore_index=True
            )
            saved.to_csv(output, index=False, encoding="utf-8-sig")

            if i % 10 == 0 or i == len(pending):
                print(f"تم إنجاز {i} من {len(pending)} تسجيل")

        del model

    print("تم حفظ النتائج في:", output)
    return saved

In [20]:
# CELL 20 | نجهز عرض WER الصحيح لكل مصدر ولكل البيانات
def show_wer(results):
    # WER الإجمالي على كل الكلمات، مو متوسط نسب التسجيلات
    for name, group in list(results.groupby("source")) + [("ALL", results)]:
        refs = [normalize_ar(x) for x in group["reference"]]
        hyps = [normalize_ar(x) for x in group["hypothesis"]]
        score = jiwer.wer(refs, hyps)
        print(f"{name}: {score:.2%}  |  تسجيلات: {len(group)}")

In [52]:
# CELL 21 | نشغّل Whisper Tiny ونشوف WER
tiny_results = run_model("tiny")
print("\nنتائج Whisper Tiny:")
show_wer(tiny_results)

التسجيلات المتبقية: 199
تم إنجاز 10 من 199 تسجيل
تم إنجاز 20 من 199 تسجيل
تم إنجاز 30 من 199 تسجيل
تم إنجاز 40 من 199 تسجيل
تم إنجاز 50 من 199 تسجيل
تم إنجاز 60 من 199 تسجيل
تم إنجاز 70 من 199 تسجيل
تم إنجاز 80 من 199 تسجيل
تم إنجاز 90 من 199 تسجيل
تم إنجاز 100 من 199 تسجيل
تم إنجاز 110 من 199 تسجيل
تم إنجاز 120 من 199 تسجيل
تم إنجاز 130 من 199 تسجيل
تم إنجاز 140 من 199 تسجيل
تم إنجاز 150 من 199 تسجيل
تم إنجاز 160 من 199 تسجيل
تم إنجاز 170 من 199 تسجيل
تم إنجاز 180 من 199 تسجيل
تم إنجاز 190 من 199 تسجيل
تم إنجاز 199 من 199 تسجيل
تم حفظ النتائج في: c:\Users\user\Desktop\CallSense\results\whisper\whisper_tiny.csv

نتائج Whisper Tiny:
CommonVoice: 91.63%  |  تسجيلات: 50
MGB2: 70.75%  |  تسجيلات: 100
Team: 90.14%  |  تسجيلات: 49
ALL: 86.03%  |  تسجيلات: 199


#  Whisper Base

وش بنسوي؟ بنستخدم نفس الملفات ونفس لغة التفريغ (ar) ونفس طريقة حساب WER، لكن بنبدّل النموذج إلى Base.

ليش؟ عشان نعرف هل النموذج الأكبر من Tiny يعطي تفريغ أدق على عيناتنا، مو على بيانات ثانية.

In [53]:
# CELL 22 | نشغّل Whisper Base على نفس التسجيلات
base_results = run_model("base")
print("✅ انتهى Whisper Base")

التسجيلات المتبقية: 199


100%|███████████████████████████████████████| 139M/139M [00:08<00:00, 16.9MiB/s]


تم إنجاز 10 من 199 تسجيل
تم إنجاز 20 من 199 تسجيل
تم إنجاز 30 من 199 تسجيل
تم إنجاز 40 من 199 تسجيل
تم إنجاز 50 من 199 تسجيل
تم إنجاز 60 من 199 تسجيل
تم إنجاز 70 من 199 تسجيل
تم إنجاز 80 من 199 تسجيل
تم إنجاز 90 من 199 تسجيل
تم إنجاز 100 من 199 تسجيل
تم إنجاز 110 من 199 تسجيل
تم إنجاز 120 من 199 تسجيل
تم إنجاز 130 من 199 تسجيل
تم إنجاز 140 من 199 تسجيل
تم إنجاز 150 من 199 تسجيل
تم إنجاز 160 من 199 تسجيل
تم إنجاز 170 من 199 تسجيل
تم إنجاز 180 من 199 تسجيل
تم إنجاز 190 من 199 تسجيل
تم إنجاز 199 من 199 تسجيل
تم حفظ النتائج في: c:\Users\user\Desktop\CallSense\results\whisper\whisper_base.csv
✅ انتهى Whisper Base


In [54]:
# CELL 23 | نعرض WER الخاص بـWhisper Base
print("نتائج Whisper Base:")
show_wer(base_results)

نتائج Whisper Base:
CommonVoice: 85.71%  |  تسجيلات: 50
MGB2: 53.36%  |  تسجيلات: 100
Team: 80.27%  |  تسجيلات: 49
ALL: 74.67%  |  تسجيلات: 199


#  Whisper Small

الآن بنجرب نموذج Whisper Small على نفس التسجيلات الـ199 وبنفس الإعدادات المستخدمة في Tiny وBase.

ليش؟ عشان نعرف هل زيادة حجم النموذج تساعد في تقليل أخطاء التفريغ، خصوصًا مع اختلاف الفصحى واللهجات العربية.

 قد يستغرق تشغيل Small وقتًا أطول لأنه أكبر من النموذجين السابقين.

In [55]:
# CELL 24 | نشغّل Whisper Small على نفس التسجيلات
small_results = run_model("small")
print("✅ انتهى Whisper Small")

التسجيلات المتبقية: 199


100%|███████████████████████████████████████| 461M/461M [00:37<00:00, 12.9MiB/s]


تم إنجاز 10 من 199 تسجيل
تم إنجاز 20 من 199 تسجيل
تم إنجاز 30 من 199 تسجيل
تم إنجاز 40 من 199 تسجيل
تم إنجاز 50 من 199 تسجيل
تم إنجاز 60 من 199 تسجيل
تم إنجاز 70 من 199 تسجيل
تم إنجاز 80 من 199 تسجيل
تم إنجاز 90 من 199 تسجيل
تم إنجاز 100 من 199 تسجيل
تم إنجاز 110 من 199 تسجيل
تم إنجاز 120 من 199 تسجيل
تم إنجاز 130 من 199 تسجيل
تم إنجاز 140 من 199 تسجيل
تم إنجاز 150 من 199 تسجيل
تم إنجاز 160 من 199 تسجيل
تم إنجاز 170 من 199 تسجيل
تم إنجاز 180 من 199 تسجيل
تم إنجاز 190 من 199 تسجيل
تم إنجاز 199 من 199 تسجيل
تم حفظ النتائج في: c:\Users\user\Desktop\CallSense\results\whisper\whisper_small.csv
✅ انتهى Whisper Small


In [56]:
# CELL 25 | نعرض WER الخاص بـWhisper Small
print("نتائج Whisper Small:")
show_wer(small_results)

نتائج Whisper Small:
CommonVoice: 55.17%  |  تسجيلات: 50
MGB2: 34.39%  |  تسجيلات: 100
Team: 65.93%  |  تسجيلات: 49
ALL: 58.77%  |  تسجيلات: 199


# Error Analysis تحليل الأخطاء

في هذا القسم، نحلل نتائج نماذج Whisper الثلاثة (Tiny وBase وSmall) باستخدام التسجيلات الـ199 نفسها.

نقارن نسب الأخطاء (WER) لكل نموذج، ونحدد أنواع الأخطاء مثل استبدال الكلمات (Substitution)، وحذفها (Deletion)، وإضافة كلمات غير موجودة (Insertion).

الهدف هو معرفة النموذج الأكثر دقة وفهم أسباب اختلاف أدائه بين مصادر البيانات الثلاثة، مع حفظ نتائج المقارنة في ملف CSV.

In [21]:
# CELL 26 | قراءة نتائج النماذج الثلاثة

model_results = {}

for name in ["tiny", "base", "small"]:
    path = RESULTS_DIR / f"whisper_{name}.csv"
    result = pd.read_csv(path)

    if len(result) != len(data):
        raise ValueError(f"عدد التسجيلات غير متطابق في {name}")

    model_results[name] = data.merge(
        result[["row_key", "hypothesis"]],
        on="row_key"
    )

    print(f" {name}: {len(result)} تسجيل")
    #في هذاالكود نتأكد ان عدد التسجيلات متطابق لكل النماذج

 tiny: 199 تسجيل
 base: 199 تسجيل
 small: 199 تسجيل


In [22]:
# CELL 27 | مقارنة النماذج

summary_rows = []

for model, result in model_results.items():
    for source in ["CommonVoice", "MGB2", "Team", "ALL"]:
        group = result if source == "ALL" else result[result["source"] == source]

        counts = jiwer.process_words(
            [normalize_ar(x) for x in group["reference"]],
            [normalize_ar(x) for x in group["hypothesis"]]
        )

        summary_rows.append({
            "Model": model,
            "Source": source,
            "Samples": len(group),
            "WER (%)": round(counts.wer * 100, 2),
            "Substitutions": counts.substitutions,
            "Deletions": counts.deletions,
            "Insertions": counts.insertions
        })

summary = pd.DataFrame(summary_rows)
display(summary)

summary.to_csv(RESULTS_DIR / "wer_comparison.csv", index=False, encoding="utf-8-sig")

,Model,Source,Samples,WER (%),Substitutions,Deletions,Insertions
0,tiny,CommonVoice,50,91.63,135,38,13
1,tiny,MGB2,100,70.75,696,131,68
2,tiny,Team,49,90.14,2757,1192,30
3,tiny,ALL,199,86.03,3588,1361,111
4,base,CommonVoice,50,85.71,136,16,22
5,base,MGB2,100,53.36,557,83,35
6,base,Team,49,80.27,2792,627,124
7,base,ALL,199,74.67,3485,726,181
8,small,CommonVoice,50,55.17,91,12,9
9,small,MGB2,100,34.39,326,73,36


In [23]:
# CELL 28 | مقارنة النماذج وتحديد الأفضل

comparison = summary.pivot(
    index="Source",
    columns="Model",
    values="WER (%)"
)

comparison = comparison.reindex(["CommonVoice", "MGB2", "Team", "ALL"])
display(comparison)

best_model = summary[summary["Source"] == "ALL"].sort_values("WER (%)").iloc[0]["Model"]

print("أفضل نموذج حسب WER:", best_model)
print("ملاحظة: هذا التقييم للدقة فقط، ولا يقارن السرعة أو التكلفة.")

Model,base,small,tiny
Source,,,
CommonVoice,85.71,55.17,91.63
MGB2,53.36,34.39,70.75
Team,80.27,65.93,90.14
ALL,74.67,58.77,86.03


أفضل نموذج حسب WER: small
ملاحظة: هذا التقييم للدقة فقط، ولا يقارن السرعة أو التكلفة.


# عرض أمثلة على أخطاء التفريغ

في هذا القسم، نختار أعلى 3 تسجيلات في نسبة الخطأ (WER) لكل نموذج من نماذج Whisper الثلاثة: Tiny وBase وSmall.

نعرض النص الأصلي والتفريغ الناتج عن كل نموذج للمقارنة بينهما وتوضيح الأخطاء.

In [24]:
# CELL 29 | أسوأ 3 تسجيلات لكل نموذج

examples = []

for model in model_results:
    result = model_results[model].copy()

    result["WER (%)"] = [
        round(jiwer.wer(normalize_ar(a), normalize_ar(b)) * 100, 2)
        for a, b in zip(result["reference"], result["hypothesis"])
    ]

    result["Model"] = model
    examples.append(result.nlargest(3, "WER (%)"))

error_examples = pd.concat(examples, ignore_index=True)

display(error_examples[["Model", "source", "row_key", "reference", "hypothesis", "WER (%)"]])

error_examples.to_csv(
    RESULTS_DIR / "error_examples.csv",
    index=False,
    encoding="utf-8-sig"
)

,Model,source,row_key,reference,hypothesis,WER (%)
0,tiny,MGB2,cd7078c8bc77e1c8c96ea94ec1ef46cd0ed574d4f4782a...,ولما حدثت كامب ديفد سبعة وسبعين أيوه,لا يقلقِ وخيرك matureع kró vis Passion لكل كاس...,257.14
1,tiny,CommonVoice,63f0732e52200ab51877883915ffc4cd269df997ca6343...,لست مشغولاً.,نسكن أن في حق,200.00
2,tiny,CommonVoice,5127dc11b3308734de2d507c3b576f9db0e0b0acf36362...,.أستخدم الحاسوب,هاتك إلي بعض الشيء,200.00
3,base,CommonVoice,d7f13efd7016e19c5264c701289ed5df433a7beb6ad74e...,أي ساعة تنام ؟,لا ي alle تصنر do the bros,233.33
4,base,CommonVoice,14fc5796521fbd00ec7c21024e4bde6cc4ecf9892944e1...,الزجاجة ملئى بالماء.,أ raises من و will never.,200.00
5,base,CommonVoice,522503aa0442c28204f57d22c3b3080642c3c86911411a...,المكيّف لا يعمل.,عن Lux<|be|> бер علي لك,200.00
6,small,CommonVoice,7fd6949d4b4f703dbe168e2e1d1c1a0c261f63733f1ff9...,.إصطحب أخته,مستهبة ع fireplace,150.00
7,small,CommonVoice,599bdb5d8682b8dd33db299de9c9d12aaae55976bf29f4...,الأسود لي.,الله سبحانه وتعالى,150.00
8,small,CommonVoice,cd4efe3cdd9d1d57f6a1ef34b0e47cd69c3d87c2fd89d0...,ذاك قاموسي.,زيكة قاموا فيه,150.00


# تحليل أنواع الأخطاء الأكثر تكرارًا

في هذا القسم، نحلل أخطاء النموذج الأفضل أداءً حسب نسبة WER الإجمالية، وهي Whisper Small في نتائجنا.

نحسب عدد الكلمات المستبدلة والمحذوفة والمضافة، ونستعرض أكثر 10 استبدالات تكرارًا بعد تطبيق التطبيع العربي. يساعدنا ذلك على فهم طبيعة أخطاء التفريغ الصوتي.

In [25]:
# CELL 30 | تحليل أخطاء النموذج الأفضل

best_results = model_results[best_model]

refs = [normalize_ar(x) for x in best_results["reference"]]
hyps = [normalize_ar(x) for x in best_results["hypothesis"]]

word_output = jiwer.process_words(refs, hyps)
substitutions, insertions, deletions = jiwer.collect_error_counts(word_output)

print("أفضل نموذج:", best_model)
print("\nأكثر 10 استبدالات تكرارًا:")

for words, count in sorted(
    substitutions.items(),
    key=lambda item: item[1],
    reverse=True
)[:10]:
    print(f"{words[0]} ← {words[1]}: {count} مرات")

print("\nإجمالي الاستبدالات:", word_output.substitutions)
print("إجمالي الكلمات المحذوفة:", word_output.deletions)
print("إجمالي الكلمات المضافة:", word_output.insertions)

أفضل نموذج: small

أكثر 10 استبدالات تكرارًا:
Agent ← اهلا: 3 مرات
معاك ← معك: 3 مرات
الف ← 1992: 2 مرات
مين ← من: 2 مرات
هيقدر ← حيادر: 2 مرات
العملاء ← العملك: 2 مرات
Customer ← عندك: 2 مرات
الفاتوره ← الفاتورة: 2 مرات
Customer ← اه: 2 مرات
مشكله ← مشكلة: 2 مرات

إجمالي الاستبدالات: 2635
إجمالي الكلمات المحذوفة: 601
إجمالي الكلمات المضافة: 221


# **الخلاصة**

تجهيز 199 تسجيلًا من ثلاثة مصادر: Common Voice وMGB-2 وتسجيلات الفريق.
مقارنة نماذج Whisper: Tiny وBase وSmall باستخدام WER.
أفضل نموذج: Small بنسبة خطأ إجمالية 58.77%، يليه Base بنسبة 74.67%، ثم Tiny بنسبة 86.03%.
تحليل أخطاء التفريغ وحفظ النتائج في ملفات CSV.

In [26]:
# CELL 31 | نعرض أسماء ملفات النتائج النهائية
print("النتائج محفوظة هنا:", RESULTS_DIR.resolve())
for path in sorted(RESULTS_DIR.glob("*.csv")):
    print("✅", path.name)

النتائج محفوظة هنا: C:\Users\user\Desktop\CallSense\results\whisper
✅ all_samples.csv
✅ error_examples.csv
✅ ready_samples.csv
✅ results_tiny.csv
✅ wer_comparison.csv
✅ whisper_base.csv
✅ whisper_small.csv
✅ whisper_tiny.csv
